# Rotating-field potentials

Tagged-particle scans adapted from the StuckGS Figure 5 calculation. These are static landscapes at **20 mT**, not switching trajectories or predictions of the transition frequency.

The potential is the particle’s trap energy plus its interactions with the other 199 particles in the ordered 10 × 10 lattice. We use nearest periodic images, without a hard radial cutoff. The simulation’s 200 μm cutoff and periodic-image conventions need an audit before a rate calculation.

All curves use the fixed energy scale $k_BT_{\mathrm{ref}}$ with $T_{\mathrm{ref}}=300$ K. They are not different noise-temperature simulations.

In [1]:
%matplotlib inline
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/py").is_dir())
sys.path.insert(0, str(REPO / "src/py"))
import rotating_potential_analysis as potential
ROOT = REPO / "data/figures/rotating-potentials"

No event loop hook running.


## Saved data
Run `src/py/rotating_potential_analysis.py` with the `mnt` environment to recompute the landscapes. This notebook loads the saved reductions and only redraws the figures.

In [2]:
saved = np.load(ROOT / "potential_profiles.npz", allow_pickle=False)
profiles = pd.DataFrame(saved["values"], columns=saved["columns"][1:])
profiles.insert(0, "orientation", saved["orientation"])
barriers = pd.read_csv(ROOT / "directed_barriers.csv")
provenance = json.loads((ROOT / "provenance.json").read_text())
print("Profile columns:", profiles.columns.tolist())
print("Rows:", len(profiles), "profile samples;", len(barriers), "barrier records")
display(profiles.head(), barriers.head())

Profile columns: ['orientation', 'alpha', 'theta_deg', 's_um', 'U_pN_um', 'U_relative_kBTref']
Rows: 148533 profile samples; 964 barrier records


  orientation  alpha  theta_deg    s_um    U_pN_um  U_relative_kBTref
0    vertical    0.0       35.0 -5.6250  34.232216        5575.550121
1    vertical    0.0       35.0 -5.6225  34.075459        5537.703924
2    vertical    0.0       35.0 -5.6200  33.919326        5500.008378
3    vertical    0.0       35.0 -5.6175  33.763817        5462.463487
4    vertical    0.0       35.0 -5.6150  33.608932        5425.069248

  orientation  alpha  ...  barrier_kBTref                status
0    vertical    0.0  ...             NaN  opposite_well_absent
1    vertical    0.0  ...             NaN  opposite_well_absent
2    vertical    0.0  ...             NaN  opposite_well_absent
3    vertical    0.0  ...             NaN  opposite_well_absent
4    vertical    0.0  ...             NaN  opposite_well_absent

[5 rows x 9 columns]

## Horizontal progress coordinate
$\alpha$ describes an imposed displacement of **all horizontal particles**: $\mathbf c_h(\alpha)=(1-2\alpha)\mathbf c_h(0)$. At $\alpha=0$ they occupy their original wells; at $\alpha=0.5$ they sit at their trap centers; at $\alpha=1$ they occupy the opposite wells.

Vertical neighbors stay at their initial positions, while one tagged vertical particle is scanned. Its coordinate $s$ points toward its initially occupied side, so the starting well is at $s/d=+1/2$. These background stages do not specify $\alpha(t)$ and need not be dynamically reachable at every field angle.

## Potential profiles
Colors give the field’s polar angle. $U_{\mathrm{rel}}=U(s)-U(d/2)$, evaluated separately for each angle and background; offsets do not alter the barrier. The inset shows the directed barrier $\tilde h=\Delta U/(k_BT_{\mathrm{ref}})$ for the initial background. Outer stiff walls are clipped to expose the inner landscape. The style follows the earlier narrow two-panel figure, top colorbar, serif text, and inset.

In [3]:
fig = potential.plot_profiles(profiles, barriers)
plt.show()

<Figure size 338.583x240.157 with 3 Axes>

## Directed barriers and saddle positions
The barrier is measured from the positive-side local minimum to the intervening maximum separating it from the opposite-side minimum. A zero with `occupied_well_absent` means the initially occupied side has no metastable minimum. A missing value with `opposite_well_absent` means there is no opposite-side minimum to define this well-to-well barrier. Neither status is a calculated transition probability.

In [4]:
fig = potential.plot_barriers(barriers)
plt.show()

<Figure size 338.583x208.661 with 2 Axes>

## What changes with horizontal motion?
At $\theta=45^\circ$, the frozen initial background gives a vertical barrier of about $259\,k_BT_{\mathrm{ref}}$. The halfway background gives about $22\,k_BT_{\mathrm{ref}}$, and the fully switched background gives about $4.4\,k_BT_{\mathrm{ref}}$. At $40^\circ$, the imposed halfway and fully switched backgrounds have no positive-side well. This shows why the background must evolve in a switching model; it does **not** establish that those backgrounds are reached at these angles.

Next, calculate a coherent horizontal pathway before assigning a time-dependent vertical potential. The existing empirical transition table remains separate.

In [5]:
selection = barriers.query("orientation == 'vertical' and theta_deg in [40,45,50,55]")
display(selection[["alpha", "theta_deg", "status", "barrier_kBTref", "saddle_um"]])
print("Verified checks:", provenance["checks"])
print("Saved figures:")
for path in sorted(ROOT.glob("*.pdf")):
    print(path)

Verified checks: {'legacy_horizontal_field_identity': True, 'analytical_force_matches_energy_derivative': True, 'periodic_translation_invariant': True, 'trap_height_and_minima': True, 'parameter_quantities_match_source': True}
Saved figures:
/home/holo/documents/mnt/frequencygs/repo/data/figures/rotating-potentials/rotating_vertical_potentials.pdf
/home/holo/documents/mnt/frequencygs/repo/data/figures/rotating-potentials/vertical_barriers_and_saddles.pdf


     alpha  theta_deg                status  barrier_kBTref  saddle_um
40     0.0       40.0             two_wells      469.484531  -4.633826
60     0.0       45.0             two_wells      258.510010  -2.648509
80     0.0       50.0             two_wells      163.972576  -1.090080
100    0.0       55.0             two_wells      122.868849   0.024796
281    0.5       40.0  occupied_well_absent        0.000000        NaN
301    0.5       45.0             two_wells       22.182415   3.035792
321    0.5       50.0             two_wells       92.720997   2.043869
341    0.5       55.0             two_wells      170.261202   1.624397
522    1.0       40.0  occupied_well_absent        0.000000        NaN
542    1.0       45.0             two_wells        4.368355   4.650857
562    1.0       50.0             two_wells      106.510008   3.353892
582    1.0       55.0             two_wells      283.241779   2.442141